# Punto 1 · Notebook 01 – Exploración de la serie y preparación para la RNN

**Taller 1 – Aprendizaje Profundo** · Daniel Sebastian Velasco Munar

En el notebook anterior (`00`) bajamos los datos de MeteoNet, revisamos la calidad de todas las estaciones de la zona NW y nos quedamos con la **estación 62548002** (cerca de Calais, en la costa norte de Francia, a 3 m sobre el nivel del mar). De ahí salió un CSV horario con 3 años de datos (2016-2018) que es lo que usamos acá.

En este notebook queremos hacer tres cosas, en orden:

1. **Entender la serie** antes de modelar: cómo se comporta la temperatura, qué tan fuertes son los ciclos diario y anual, dónde hay huecos, y qué relación tiene con las otras variables. Esto nos sirve para tomar decisiones con argumentos y no a ojo.
2. **Definir la tarea de predicción**: cuántas horas hacia atrás mira el modelo y cuántas hacia adelante predice. El taller pide dejar esto explícito.
3. **Preparar los datos** para el notebook `02`: imputar los pocos faltantes, construir las variables de calendario, partir en entrenamiento / validación / test **sin fuga de información**, y guardar todo con su configuración.

No hace falta GPU para este notebook. Se puede correr en Colab o en local.


## 0. Preparación del entorno

In [ ]:
import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

EN_COLAB = "google.colab" in sys.modules
print("Colab:", EN_COLAB)


In [ ]:
# En Colab clonamos el repo para leer el CSV de la estación y dejar las figuras en results/.
REPO_URL = "https://github.com/DANIEL-VELASCO/taller1-deep-learning.git"

if EN_COLAB:
    if not Path("/content/taller1-deep-learning").exists():
        !git clone -q {REPO_URL} /content/taller1-deep-learning
    RAIZ = Path("/content/taller1-deep-learning")
else:
    RAIZ = Path.cwd().resolve().parents[1]     # el notebook vive en code/punto1_rnn_meteonet/

ESTACION = 62548002
RUTA_DATA = RAIZ / "code" / "punto1_rnn_meteonet" / "data"
RUTA_RES = RAIZ / "results" / "punto1"
RUTA_RES.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RUTA_DATA / f"estacion_{ESTACION}_horaria.csv", index_col="date", parse_dates=True)
with open(RUTA_DATA / f"estacion_{ESTACION}_metadatos.json", encoding="utf-8") as f:
    meta = json.load(f)

print(f"{len(df):,} filas  |  {df.index.min()}  ->  {df.index.max()}")
meta


## 1. Primer vistazo

Recordatorio de qué es cada columna (ya vienen convertidas a unidades "normales" desde el notebook `00`):

- `t_c`: temperatura (°C). **Es lo que queremos predecir.**
- `td_c`: punto de rocío (°C). Muy ligado a la humedad.
- `hu`: humedad relativa (%).
- `psl_hpa`: presión a nivel del mar (hPa).
- `ff`: velocidad del viento (m/s); `u`, `v`: el mismo viento en componentes este-oeste y norte-sur, para que la dirección no tenga el salto de 359°→0°.
- `precip`: lluvia acumulada en la hora (mm).
- `n_obs`: cuántas observaciones de 6 min había en esa hora (máximo 10). Solo para control de calidad, no entra al modelo.


In [ ]:
display(df.head())
df.describe().T.round(2)


Algo que vale la pena revisar de una: la columna `n_obs`. Si casi siempre es 10, el promedio horario está bien soportado. Si hay muchas horas con 1 o 2 observaciones, esos promedios son ruidosos.

In [ ]:
print(df["n_obs"].value_counts().sort_index())
print(f"\nHoras con las 10 observaciones completas: {(df['n_obs']==10).mean():.1%}")


## 2. Valores faltantes

En el `00` vimos que a esta estación le falta el 0.09% de la temperatura a nivel horario. Es muy poco, pero hay que saber **cómo** falta: no es lo mismo 24 horas sueltas repartidas en 3 años que un día completo sin datos. Lo primero se rellena sin problema interpolando; lo segundo no se puede inventar.

In [ ]:
faltantes = df.isna().mean().sort_values(ascending=False)
print((faltantes * 100).round(3).astype(str) + " %")

# Longitud de los huecos consecutivos en la temperatura
falta_t = df["t_c"].isna()
id_hueco = (~falta_t).cumsum()[falta_t]
huecos = falta_t[falta_t].groupby(id_hueco).agg(["size"])
huecos["inicio"] = falta_t[falta_t].groupby(id_hueco).apply(lambda s: s.index.min())
huecos = huecos.rename(columns={"size": "horas"}).sort_values("horas", ascending=False).reset_index(drop=True)
print(f"\n{len(huecos)} huecos en la temperatura. Los más largos:")
huecos.head(10)


In [ ]:
fig, ax = plt.subplots(figsize=(14, 2.2))
ax.eventplot(df.index[df["t_c"].isna()].values, colors="crimson", linelengths=0.8)
ax.set_yticks([]); ax.set_title("Horas sin temperatura a lo largo de los 3 años")
ax.set_xlim(df.index.min(), df.index.max())
fig.savefig(RUTA_RES / "eda_huecos_temperatura.png", bbox_inches="tight"); plt.show()


> **Observaciones (completar después de correr):** ¿cuántos huecos hay y de qué tamaño? ¿están concentrados en una fecha o dispersos? Esto define la estrategia de imputación de la sección 6.

## 3. Cómo se comporta la temperatura

Primero la serie completa. Lo que esperamos ver: el ciclo anual (veranos e inviernos) y, por ser una estación costera, una amplitud relativamente moderada (el mar amortigua los extremos).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
df["t_c"].plot(ax=ax, lw=0.4, color="tab:blue")
df["t_c"].rolling("30D").mean().plot(ax=ax, lw=2, color="tab:red", label="media móvil 30 días")
ax.set_ylabel("°C"); ax.set_xlabel(""); ax.legend(loc="upper left")
ax.set_title(f"Temperatura horaria – estación {ESTACION} (2016-2018)")
fig.savefig(RUTA_RES / "eda_serie_completa.png", bbox_inches="tight"); plt.show()


Ahora un zoom de dos semanas, para ver el ciclo diario "de cerca". Ojo con qué tan regular es: si cada día se ve parecido al anterior, el modelo lo va a tener relativamente fácil; si hay días donde el patrón se rompe (frentes, lluvia), ahí es donde va a fallar.

In [ ]:
zoom = df.loc["2017-07-01":"2017-07-14", "t_c"]
fig, ax = plt.subplots(figsize=(14, 3.5))
zoom.plot(ax=ax, marker=".", ms=3, lw=1)
ax.set_ylabel("°C"); ax.set_xlabel(""); ax.set_title("Dos semanas de julio de 2017")
fig.savefig(RUTA_RES / "eda_zoom_dos_semanas.png", bbox_inches="tight"); plt.show()


### 3.1 Ciclo anual y ciclo diario

Los dos ciclos que dominan una serie de temperatura. El **anual** se ve agrupando por mes; el **diario** agrupando por hora del día. Y como el ciclo diario cambia con la estación (en verano la amplitud día-noche es mayor), lo dibujamos separado por estación del año.

In [ ]:
df["mes"] = df.index.month
df["hora"] = df.index.hour
estaciones_anio = {12: "invierno", 1: "invierno", 2: "invierno", 3: "primavera", 4: "primavera", 5: "primavera",
                   6: "verano", 7: "verano", 8: "verano", 9: "otoño", 10: "otoño", 11: "otoño"}
df["estacion_anio"] = df["mes"].map(estaciones_anio)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))

sns.boxplot(data=df, x="mes", y="t_c", ax=axes[0], color="lightsteelblue", fliersize=1)
axes[0].set_title("Ciclo anual: distribución de la temperatura por mes"); axes[0].set_ylabel("°C")

perfil = df.groupby(["estacion_anio", "hora"])["t_c"].mean().unstack(0)[["invierno", "primavera", "verano", "otoño"]]
perfil.plot(ax=axes[1], marker="o", ms=3)
axes[1].set_title("Ciclo diario: temperatura media por hora, según estación del año")
axes[1].set_xlabel("hora del día (UTC)"); axes[1].set_ylabel("°C"); axes[1].set_xticks(range(0, 24, 2))

fig.savefig(RUTA_RES / "eda_ciclos_anual_diario.png", bbox_inches="tight"); plt.show()

amplitud = perfil.max() - perfil.min()
print("Amplitud del ciclo diario (°C):"); print(amplitud.round(2))


> **Observaciones:** ¿cuánto vale la amplitud diaria en verano vs invierno? Comparar con el rango anual (diferencia entre mes más frío y más cálido). Esto da una idea de "cuánto hay que acertar" en cada horizonte: predecir a 24 h exige capturar bien el ciclo diario; predecir a 1 h exige muy poco.

### 3.2 Distribución

Miramos el histograma por dos razones: ver si hay valores absurdos (un sensor dañado se nota como picos raros) y saber si la variable es más o menos simétrica, que es lo que le gusta a una red con `StandardScaler`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
sns.histplot(df["t_c"].dropna(), bins=60, ax=axes[0], color="tab:blue"); axes[0].set_title("Temperatura (°C)")
sns.histplot(df["hu"].dropna(), bins=50, ax=axes[1], color="tab:green"); axes[1].set_title("Humedad (%)")
sns.histplot(df["precip"].dropna().clip(upper=5), bins=50, ax=axes[2], color="tab:gray"); axes[2].set_title("Precipitación (mm/h, recortada a 5)")
fig.savefig(RUTA_RES / "eda_distribuciones.png", bbox_inches="tight"); plt.show()

print(f"Horas con lluvia > 0: {(df['precip'] > 0).mean():.1%}")


## 4. Relación con las otras variables y "memoria" de la serie

Dos preguntas que nos importan para decidir qué entra al modelo y con cuánta historia:

1. ¿Las demás variables aportan información sobre la temperatura? (correlación)
2. ¿Cuánto "se acuerda" la temperatura de sí misma? (autocorrelación). Esto es lo que justifica la **longitud de la ventana de entrada**.

In [ ]:
variables = ["t_c", "td_c", "hu", "psl_hpa", "ff", "u", "v", "precip"]
corr = df[variables].corr()

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title("Correlación entre variables")
fig.savefig(RUTA_RES / "eda_correlacion.png", bbox_inches="tight"); plt.show()


La autocorrelación en el rezago *k* mide qué tan parecida es la temperatura de ahora a la de hace *k* horas. En una serie con ciclo diario esperamos picos en 24, 48, 72... horas. Si esos picos se mantienen altos varios días, tiene sentido darle al modelo varios días de historia.

In [ ]:
max_lag = 24 * 8
lags = np.arange(1, max_lag + 1)
acf = np.array([df["t_c"].autocorr(lag=k) for k in lags])

fig, ax = plt.subplots(figsize=(14, 3.6))
ax.plot(lags, acf, lw=1.2)
for d in range(1, 8):
    ax.axvline(24 * d, color="gray", ls=":", lw=0.8)
ax.axvspan(0, 72, color="tab:orange", alpha=0.10, label="ventana de entrada propuesta (72 h)")
ax.set_xlabel("rezago (horas)"); ax.set_ylabel("autocorrelación"); ax.set_xticks(range(0, max_lag + 1, 24))
ax.set_title("Autocorrelación de la temperatura horaria"); ax.legend(loc="lower left")
fig.savefig(RUTA_RES / "eda_autocorrelacion.png", bbox_inches="tight"); plt.show()

print("ACF en múltiplos de 24 h:", {f"{24*d} h": round(float(acf[24*d - 1]), 3) for d in range(1, 8)})


### 4.1 ¿Qué tan difícil es cada horizonte?

Antes de entrenar nada conviene saber cuánto error comete el modelo más tonto posible, la **persistencia**: "la temperatura dentro de *k* horas será igual a la de ahora". Su error crece con *k* y nos da una vara para medir a la RNN. Si la red no le gana a esto, no sirve.

In [ ]:
horizontes = np.arange(1, 49)
mae_persistencia = np.array([(df["t_c"].shift(-k) - df["t_c"]).abs().mean() for k in horizontes])

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(horizontes, mae_persistencia, marker="o", ms=3)
ax.axvline(24, color="tab:red", ls="--", lw=1, label="horizonte propuesto (24 h)")
ax.set_xlabel("horizonte k (horas)"); ax.set_ylabel("MAE de persistencia (°C)")
ax.set_title("Error del modelo de persistencia según el horizonte"); ax.legend()
fig.savefig(RUTA_RES / "eda_dificultad_horizonte.png", bbox_inches="tight"); plt.show()

for k in [1, 3, 6, 12, 24, 48]:
    print(f"k = {k:2d} h  ->  MAE persistencia = {mae_persistencia[k-1]:.2f} °C")


> **Observaciones:** el error de persistencia a 1 h es muy bajo (la temperatura casi no cambia en una hora), sube hasta ~12 h y vuelve a bajar en 24 h por el ciclo diario. Anotar los valores: son la referencia que la RNN tiene que superar en el notebook `02`.

## 5. Definición de la tarea

Con lo visto arriba, dejamos fijas las decisiones (esto va tal cual al informe):

| Decisión | Valor | Por qué |
|---|---|---|
| Variable objetivo | `t_c` (temperatura en °C) | Lo que pide el taller |
| Frecuencia | horaria | Los datos vienen cada 6 min; a esa escala hay mucho ruido y secuencias larguísimas. Con 1 h se conserva el ciclo diario con 24 puntos |
| **Longitud de entrada** | **72 h** (3 días) | La autocorrelación se mantiene alta en múltiplos de 24 h; con 3 días la red ve tres ciclos diarios y la tendencia reciente. En `02` se prueba también 24, 48 y 168 h |
| **Horizonte** | **24 h**, las 24 horas siguientes de una vez (multi-salida) | Es el problema útil ("mañana") y no es trivial: la persistencia falla bastante. Se compara además con horizonte 1 h como referencia fácil |
| Entradas del modelo | `t_c, td_c, hu, psl_hpa, ff, u, v, precip` + hora y día del año codificados en seno/coseno | Todas las variables observadas más el "reloj", para que la red sepa en qué momento del día y del año está |
| Test | **último 30%** de la serie, en orden cronológico | Lo exige el taller; además es la única forma honesta de evaluar una serie de tiempo |
| Validación | último 15% del 70% restante | Para elegir hiperparámetros y activar los callbacks sin tocar el test |

Esquema de una muestra de entrenamiento:

```
horas:   ... [ h-71  h-70  ...  h-1   h ] [ h+1  h+2  ...  h+24 ] ...
               └──── entrada: 72 h ─────┘  └── salida: 24 h ───┘
               12 variables por hora        solo temperatura
```

Deslizando la ventana una hora cada vez salen unas 26 000 muestras, que se reparten en los tres conjuntos **después** de cortar la serie (nunca una ventana cruza de un conjunto a otro).


## 6. Preparación de los datos

### 6.1 Imputación de faltantes

Regla simple y conservadora: los huecos cortos (hasta 6 horas seguidas) se rellenan con interpolación lineal en el tiempo; los huecos más largos se dejan como `NaN` y en el `02` esas ventanas se descartan. Además marcamos con una bandera qué horas de temperatura fueron imputadas, para poder **excluirlas de las métricas de test**: no tiene sentido evaluar el modelo contra un valor que nos inventamos.

In [ ]:
columnas_modelo = ["t_c", "td_c", "hu", "psl_hpa", "ff", "u", "v", "precip"]
LIMITE_INTERP = 6   # horas

original = df[columnas_modelo].copy()
interpolado = original.interpolate(method="time", limit_area="inside")

# Solo aceptamos la interpolación en huecos de hasta LIMITE_INTERP horas seguidas.
# (el parámetro `limit` de pandas no sirve para esto: rellena las primeras N horas de un hueco largo y deja el resto)
def largo_del_hueco(col):
    falta = col.isna()
    ids = (~falta).cumsum()
    return falta.groupby(ids).transform("sum").where(falta, 0)

for col in columnas_modelo:
    rellenar = original[col].isna() & (largo_del_hueco(original[col]) <= LIMITE_INTERP)
    df[col] = original[col].where(~rellenar, interpolado[col])

df["t_imputada"] = original["t_c"].isna() & df["t_c"].notna()

print(f"Horas de temperatura imputadas: {int(df['t_imputada'].sum())}")
print(f"Horas de temperatura que siguen vacías (huecos > {LIMITE_INTERP} h): {int(df['t_c'].isna().sum())}")
print("\nFaltantes restantes por columna:")
print(df[columnas_modelo].isna().sum())


### 6.2 Variables de calendario

Una red no sabe que la hora 23 y la hora 0 son vecinas. Codificar la hora (y el día del año) como seno y coseno resuelve eso: el "reloj" queda como un punto que da vueltas en un círculo, sin saltos.

In [ ]:
horas_dia = 24
dias_anio = 365.25
t_horas = df.index.hour + df.index.minute / 60
dia = df.index.dayofyear + t_horas / 24

df["hora_sin"] = np.sin(2 * np.pi * t_horas / horas_dia)
df["hora_cos"] = np.cos(2 * np.pi * t_horas / horas_dia)
df["dia_sin"] = np.sin(2 * np.pi * dia / dias_anio)
df["dia_cos"] = np.cos(2 * np.pi * dia / dias_anio)

fig, ax = plt.subplots(figsize=(10, 2.6))
df.loc["2017-01-01":"2017-01-03", ["hora_sin", "hora_cos"]].plot(ax=ax)
ax.set_title("Codificación de la hora del día (3 días de ejemplo)"); ax.set_xlabel("")
plt.show()

FEATURES = columnas_modelo + ["hora_sin", "hora_cos", "dia_sin", "dia_cos"]
print(len(FEATURES), "variables de entrada:", FEATURES)


### 6.3 Partición cronológica: entrenamiento / validación / test

Acá está el punto delicado de la rúbrica ("evitando fuga de información"). Cortamos la serie por **fechas**, en orden: lo primero es entrenamiento, luego validación y el 30% final es test. Nada de barajar. Y guardamos los índices de corte para que el `02` use exactamente la misma partición.

In [ ]:
n = len(df)
i_test = int(n * 0.70)          # de aquí en adelante es test (30% final)
i_val = int(i_test * 0.85)      # dentro del 70%: 85% entrenamiento, 15% validación

conjuntos = {
    "entrenamiento": df.iloc[:i_val],
    "validacion":    df.iloc[i_val:i_test],
    "test":          df.iloc[i_test:],
}
for nombre, parte in conjuntos.items():
    print(f"{nombre:14s} {len(parte):6,} h  ({len(parte)/n:5.1%})   {parte.index.min()}  ->  {parte.index.max()}")

fig, ax = plt.subplots(figsize=(14, 3.5))
colores = {"entrenamiento": "tab:blue", "validacion": "tab:orange", "test": "tab:green"}
for nombre, parte in conjuntos.items():
    parte["t_c"].plot(ax=ax, lw=0.4, color=colores[nombre], label=nombre)
ax.set_ylabel("°C"); ax.set_xlabel(""); ax.legend(loc="upper left", ncol=3)
ax.set_title("Partición cronológica de la serie")
fig.savefig(RUTA_RES / "eda_particion.png", bbox_inches="tight"); plt.show()


Una consecuencia de partir así que vale la pena decir en el informe: el test va de **febrero a diciembre de 2018**, así que el modelo se evalúa sobre una primavera, un verano y un otoño que nunca vio, pero solo unas tres semanas de invierno. Es una limitación de tener solo 3 años.

### 6.4 Estadísticas para escalar

Las redes entrenan mejor con entradas centradas y con varianza parecida. Usaremos `(x - media) / desviación` por variable, pero **la media y la desviación se calculan solo con el conjunto de entrenamiento**. Si usáramos toda la serie, el modelo "sabría" algo del test aunque sea de forma indirecta. Las guardamos en la configuración para que el `02` no tenga que recalcularlas.

In [ ]:
stats_train = df.iloc[:i_val][FEATURES].agg(["mean", "std"]).T
stats_train.round(3)


## 7. Guardar todo para el notebook `02`

Dos archivos:

- `estacion_62548002_preparada.csv`: la serie horaria imputada, con las variables de calendario y la bandera de imputación.
- `config_tarea.json`: la definición de la tarea (ventana, horizonte, variables, índices de corte, estadísticas de escalado). Así el `02` no repite decisiones ni puede desalinearse de lo que se decidió acá.

In [ ]:
columnas_guardar = FEATURES + ["t_imputada", "n_obs"]
RUTA_PREP = RUTA_DATA / f"estacion_{ESTACION}_preparada.csv"
df[columnas_guardar].round(4).to_csv(RUTA_PREP)

config = {
    "estacion": ESTACION,
    "objetivo": "t_c",
    "features": FEATURES,
    "largo_entrada_h": 72,
    "horizonte_h": 24,
    "horizonte_referencia_h": 1,
    "frecuencia": "1h",
    "i_val": int(i_val),
    "i_test": int(i_test),
    "fechas": {k: [str(v.index.min()), str(v.index.max())] for k, v in conjuntos.items()},
    "limite_interpolacion_h": LIMITE_INTERP,
    "escalado": {"media": stats_train["mean"].round(6).to_dict(), "std": stats_train["std"].round(6).to_dict()},
    "mae_persistencia_serie_completa": {str(k): round(float(mae_persistencia[k-1]), 4) for k in [1, 3, 6, 12, 24, 48]},
}
with open(RUTA_DATA / "config_tarea.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f"guardado {RUTA_PREP.name} ({RUTA_PREP.stat().st_size/1e6:.2f} MB)")
print("guardado config_tarea.json")
print("\nfiguras en results/punto1/:")
for p in sorted(RUTA_RES.glob("eda_*.png")):
    print("  ", p.name)


## 8. Qué nos llevamos de aquí para el informe

Resumen de lo que este notebook aporta a cada parte del reporte (completar con los números que salieron):

- **Selección de estación (con el `00`)**: estación 62548002, costera, 3 años completos, 0.09 % de faltantes en temperatura, todas las variables disponibles incluida presión.
- **Diseño experimental**: partición cronológica 59.5 % / 10.5 % / 30 % (train / val / test), fechas exactas en `config_tarea.json`; escalado ajustado solo con entrenamiento; imputación solo de huecos ≤ 6 h y horas imputadas excluidas de las métricas.
- **Formulación de la tarea**: entrada 72 h × 12 variables → salida 24 h de temperatura (más un horizonte de 1 h como referencia). Justificada con la autocorrelación y la curva de dificultad por horizonte.
- **Línea base**: MAE de persistencia a 24 h = ___ °C (y a 1 h = ___ °C). La RNN tiene que quedar por debajo.
- **Limitación a mencionar**: el test casi no incluye invierno (solo febrero de 2018).
- **Ojo con la línea base a 24 h**: como el modelo predice las 24 horas siguientes de una vez, la persistencia justa se calcula promediando los horizontes 1 a 24, no solo el de 24 h. Eso se hace en el `02` sobre las mismas ventanas de test.

Archivos que genera este notebook (si se corre en Colab, descargarlos del panel de archivos y subirlos al repo):
`data/estacion_62548002_preparada.csv`, `data/config_tarea.json` y las figuras `results/punto1/eda_*.png`.
